# P5 エージェントを結果と軌跡で測る

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

クロツグ食品（架空）の発注エージェントを、素の SDK の道具呼び出しループで動かし、発注の中身（結果）と、道具の呼び方（軌跡）の両方で採点します。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 画面に収める小さな道具
長い行を画面の幅で折り返して表示するだけの関数です（中身は変えません）。

In [ ]:
import io, contextlib
def fit(fn, *a, w=104, **k):   # fn の表示を、全角 2・半角 1 で数えた幅 w で折り返す
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn(*a, **k)
    for line in buf.getvalue().splitlines():
        out, n = "", 0
        for ch in line:
            d = 2 if ord(ch) > 0x2E80 else 1
            if n + d > w:
                print(out)
                out, n = "        ", 8
            out, n = out + ch, n + d
        print(out)

## 1. 発注の手引きと、エージェントが使える道具
手引きは `data/agent.json`、道具は `lab05_agent_trajectory/env.py` の本物です。

In [ ]:
from common import kit, llm as L, metrics as M, stats
ag = lab("agent", "lab05_agent_trajectory")
from env import TOOLS
data = kit.load_json("agent.json")
fit(print, data["rules"].strip())
print("\n道具:", " / ".join(t["name"] for t in TOOLS))
print("依頼:", len(data["tasks"]), "件。例", data["tasks"][3]["id"], data["tasks"][3]["request"])

## 2. 依頼 T04 を 1 回動かし、軌跡を見る
Opus 5.5 の本物の応答を記録から再生します（費用 0、費用と待ち時間は元の実行の値）。

In [ ]:
P = "replay" if PROVIDER == "mock" else PROVIDER
t04 = next(t for t in data["tasks"] if t["id"] == "T04")
with safe():
    ep = ag.run_episode(t04, data, provider=P, model="claude-opus-5-5", trial=0)
    for i, s in enumerate(ep["trace"], 1):
        print(f"{i}. {s['tool']}({', '.join(f'{k}={v}' for k, v in s['args'].items())})")
        print(f"     → {str(s['result'])[:86]}")
    print("発注:", ep["orders"])
    print(f"往復 {ep['turns']} 回 / 元の費用 {ep['cost_usd']:.4f} USD / {ep['seconds']:.1f} 秒")

## 3. 採点のコード ― 結果と軌跡を別々に見る
`lab05_agent_trajectory/score.py` の本物です。発注の中身が合っていても、手順の違反があれば不合格にします。

In [ ]:
sc = lab("score", "lab05_agent_trajectory")
fit(show, sc.score, 1, 20)

## 4. T04 を採点し、承認を抜いた軌跡と比べる
本物の軌跡から request_approval だけを取り除いた写しを作り、同じ採点にかけます。

In [ ]:
import copy
def brief(s):
    return f"結果 {'○' if s['outcome_ok'] else '×'} / 軌跡 {'○' if s['trajectory_ok'] else '×'} / 合格 {s['pass']} / 道具 {s['tool_calls']} 回"
print("本物の軌跡      :", brief(sc.score(ep, t04, data)))
bad = copy.deepcopy(ep)
bad["trace"] = [s for s in bad["trace"] if s["tool"] != "request_approval"]
s2 = sc.score(bad, t04, data)
print("承認を抜いた写し:", brief(s2))
print("   違反:", s2["violations"])

## 5. サイコロで考える ― 1 回でも当たるか、毎回当たるか
1〜4 の目を「成功」（2/3）とし、3 回振るのを 1 万回くり返します。

In [ ]:
import random
rng = random.Random(1)
trials = [[rng.randint(1, 6) <= 4 for _ in range(3)] for _ in range(10000)]
any3 = sum(any(t) for t in trials) / len(trials)
all3 = sum(all(t) for t in trials) / len(trials)
print(f"1 回の成功率 2/3 = {2 / 3:.3f}")
print(f"3 回のうち 1 回でも成功: サイコロ 1 万回 {any3:.3f} / 式 1-(1/3)^3 = {1 - (1 / 3) ** 3:.3f}")
print(f"3 回とも成功          : サイコロ 1 万回 {all3:.3f} / 式 (2/3)^3   = {(2 / 3) ** 3:.3f}")

## 6. pass@k と pass^k の式
`common/metrics.py` の本物です。pass@k は Chen et al. 2021 の不偏推定、pass^k は τ-bench の「k 回とも成功」。

In [ ]:
fit(show, M.pass_at_k)
print()
fit(show, M.pass_hat_k)

## 7. 3 回試したときの成功回数ごとに、二つの指標を並べる

In [ ]:
rows = []
for c in range(4):
    rows.append([f"{c}/3", f"{M.pass_at_k(3, c, 1):.2f}", f"{M.pass_at_k(3, c, 2):.2f}", f"{M.pass_hat_k(3, c, 2):.2f}",
                 f"{M.pass_at_k(3, c, 3):.2f}", f"{M.pass_hat_k(3, c, 3):.2f}"])
print(kit.table(rows, ["成功", "pass@1", "pass@2", "pass^2", "pass@3", "pass^3"]))
print("本物の実行（10 件 × 3 回）は、Sonnet も Opus も全部の依頼が 3/3 → pass@3 も pass^3 も 1.00")

## 8. 10 件 × 3 回を 2 つのモデルで ― 合格・道具の数・費用・待ち時間
講師が API で回した 60 回の記録を再生します。

In [ ]:
by = {t["id"]: t for t in data["tasks"]}
best = sum(sc.minimal_calls(t) for t in data["tasks"]) * 3
EPS = {}
for model in ("claude-sonnet-5-5", "claude-opus-5-5"):
    with safe():
        eps = EPS[model] = [ag.run_episode(t, data, provider=P, model=model, trial=k) for t in data["tasks"] for k in range(3)]
        sco = [sc.score(e, by[e["task"]], data) for e in eps]
        c = {t: sum(s["pass"] for e, s in zip(eps, sco) if e["task"] == t) for t in by}
        cost, secs = sum(e["cost_usd"] for e in eps), [e["seconds"] for e in eps]
        print(f"{model:<18} 合格 {sum(s['pass'] for s in sco)}/{len(eps)}  pass^3 {sum(M.pass_hat_k(3, v, 3) for v in c.values()) / len(c):.2f}"
              f"  道具 {sum(s['tool_calls'] for s in sco)} 回（最短 {best}）")
        print(f"{'':<18} 費用 {cost:.4f} USD・1 回 {cost / len(eps):.4f}  待ち時間 p50 {stats.percentile(secs, .5):.1f}s"
              f" p95 {stats.percentile(secs, .95):.1f}s")

## 9. 依頼ごとの道具の数 ― 最短より多く呼んでいる

In [ ]:
rows = []
for t in data["tasks"]:
    avg = {m: sum(len(e["trace"]) for e in EPS.get(m, []) if e["task"] == t["id"]) / 3 for m in EPS}
    rows.append([t["id"], sc.minimal_calls(t)] + [f"{avg[m]:.1f}" for m in EPS] + [t["request"][:24]])
print(kit.table(rows, ["依頼", "最短", "Sonnet", "Opus", "依頼の頭"]))

## 10. 失敗 0 件の読み方 ― 30 回中 0 回でも、失敗率の上限は 0 ではない

In [ ]:
lo, hi = stats.wilson(30, 30)
print(f"30 回中 30 回合格 → 合格率の 95% 区間 [{100 * lo:.1f}, {100 * hi:.1f}]")
print(f"   → 失敗率の上限 {100 * (1 - lo):.1f}%（3 の法則なら 3/30 = {100 * stats.rule_of_three(30):.1f}%）")
for n in (100, 300, 1000):
    print(f"{n:>5} 回中 0 回の失敗なら、上限 {100 * (1 - stats.wilson(n, n)[0]):.1f}%")

## 11. 模擬（mock）で崩れ方を見る
🔴 ここは模擬です。決まった筋書きで、腕前の確率で承認を飛ばしたり期限を見落としたりします。数字は性能の証拠になりません。

In [ ]:
ra = lab("run_agent", "lab05_agent_trajectory")
print("🔴 模擬（mock）の結果。本物のモデルの性能ではありません")
with L.mock_handler(ra.mock_for(data)):
    for model in ("claude-sonnet-5-5", "claude-haiku-4-5"):
        eps = [ag.run_episode(t, data, provider="mock", model=model, trial=k) for t in data["tasks"] for k in range(3)]
        sco = [sc.score(e, by[e["task"]], data) for e in eps]
        c = {t: sum(s["pass"] for e, s in zip(eps, sco) if e["task"] == t) for t in by}
        print(f"模擬 {model}: 合格 {sum(c.values())}/30  pass@3 {sum(M.pass_at_k(3, v, 3) for v in c.values()) / 10:.2f}"
              f"  pass^3 {sum(M.pass_hat_k(3, v, 3) for v in c.values()) / 10:.2f}  結果だけ合格 "
              f"{sum(s['outcome_ok'] and not s['trajectory_ok'] for s in sco)}")
        for e, s in zip(eps, sco):
            if not s["pass"]:
                print(f"   {e['task']} {e['trial'] + 1} 回目: {(s['violations'] or ['発注の中身が違う'])[0]}")

## 12. 例題 ― 3 回中 2 回成功のとき、二つの指標を手で出す

In [ ]:
from math import comb
n, c = 3, 2
print(f"pass@1 = 1 - C(1,1)/C(3,1) = 1 - {comb(1, 1)}/{comb(3, 1)} = {1 - comb(1, 1) / comb(3, 1):.3f}   関数 {M.pass_at_k(n, c, 1):.3f}")
print(f"pass@2 = 1 - C(1,2)/C(3,2) = 1 - {comb(1, 2)}/{comb(3, 2)} = {1 - comb(1, 2) / comb(3, 2):.3f}   関数 {M.pass_at_k(n, c, 2):.3f}")
print(f"pass^2 = C(2,2)/C(3,2) = {comb(2, 2)}/{comb(3, 2)} = {comb(2, 2) / comb(3, 2):.3f}       関数 {M.pass_hat_k(n, c, 2):.3f}")
print(f"pass@3 = 1 - C(1,3)/C(3,3) = 1 - {comb(1, 3)}/{comb(3, 3)} = {1 - comb(1, 3) / comb(3, 3):.3f}   関数 {M.pass_at_k(n, c, 3):.3f}")
print(f"pass^3 = C(2,3)/C(3,3) = {comb(2, 3)}/{comb(3, 3)} = {comb(2, 3) / comb(3, 3):.3f}       関数 {M.pass_hat_k(n, c, 3):.3f}")
print(f"1 回の成功率を 2/3 とみなすと: 3 回のうち 1 回でも {1 - (1 / 3) ** 3:.3f} / 3 回とも {(2 / 3) ** 3:.3f}")